In [1]:
import os
import random
import sqlite3
import pandas as pd
from datetime import datetime, timedelta


pd.set_option('display.float_format', lambda x: f"{int(x):,}".replace(",", "."))

# Configuración del entorno
DB_PATH = "auditoria_ecommerce_avanzado.db"
random.seed(42) # Fijamos una semilla para que los datos aleatorios sean consistentes

provincias = ["Buenos Aires", "Córdoba", "Santa Fe", "Mendoza", "Tucumán", "Salta", "Chaco", "Neuquén", "Río Negro", "Entre Ríos"]
productos_tech = [
    {"nombre": "Notebook I7 Pro", "precio": 1450000},
    {"nombre": "Monitor Gamer 27'", "precio": 420000},
    {"nombre": "Teclado Mecánico RGB", "precio": 95000},
    {"nombre": "Mouse Óptico Inalámbrico", "precio": 55000},
    {"nombre": "Auriculares Cancelación Ruido", "precio": 180000},
    {"nombre": "Placa de Video RTX 4060", "precio": 680000},
    {"nombre": "Disco Sólido SSD 1TB", "precio": 110000}
]

# 1. GENERAR BASE DE DATOS DE 50 CLIENTES NACIONALES
clientes_lista = []
for i in range(1, 51):
    clientes_lista.append({
        "cliente_id": 100 + i,
        "nombre_cliente": f"Cliente Anónimo {100 + i}",
        "provincia": random.choice(provincias),
        "es_vip": random.choice([0, 0, 0, 1]) # 25% de probabilidad de ser VIP
    })

# 2. GENERAR HISTÓRICO DE VENTAS CON ENVÍOS FEDERALES
ventas_lista = []
fecha_base = datetime(2026, 8, 1)

for i in range(1, 120): # 119 transacciones aleatorias
    prod = random.choice(productos_tech)
    cantidad = random.randint(1, 3)
    monto_total = prod["precio"] * cantidad
    
    # Generamos un número de envío simulado tipo Andreani u OCA (ej: NV-94820-AR)
    num_envio = f"NV-{random.randint(10000, 99999)}-AR"
    fecha_venta = (fecha_base + timedelta(days=random.randint(0, 13))).strftime('%Y-%m-%d')
    
    ventas_lista.append({
        "transaccion_id": 5000 + i,
        "cliente_id": random.randint(101, 150), # Conecta con los IDs de clientes anteriores
        "producto_comprado": prod["nombre"],
        "cantidad_unidades": cantidad,
        "monto_total_ars": monto_total,
        "numero_envio": num_envio,
        "fecha_compra": fecha_venta
    })

# Guardamos los conjuntos de datos en archivos CSV físicos en tu disco (Fuera de la RAM de ejecución)
pd.DataFrame(clientes_lista).to_csv("clientes_crudo.csv", index=False)
pd.DataFrame(ventas_lista).to_csv("ventas_crudo.csv", index=False)

print(f"✓ Archivos base creados físicamente en tu carpeta: 'clientes_crudo.csv' y 'ventas_crudo.csv'")
print(f"✓ Destino final de persistencia configurado: {DB_PATH}")


✓ Archivos base creados físicamente en tu carpeta: 'clientes_crudo.csv' y 'ventas_crudo.csv'
✓ Destino final de persistencia configurado: auditoria_ecommerce_avanzado.db


In [2]:
print("Leyendo archivos CSV desde el disco local...")
print("=" * 75)

# Simulamos la ingesta real leyendo los archivos físicos del almacenamiento
df_clientes_local = pd.read_csv("clientes_crudo.csv")
df_ventas_local = pd.read_csv("ventas_crudo.csv")

# Proceso de normalización analítica y tipos de datos
df_ventas_local['monto_total_ars'] = pd.to_numeric(df_ventas_local['monto_total_ars'], errors='coerce')
df_ventas_local['cantidad_unidades'] = pd.to_numeric(df_ventas_local['cantidad_unidades'], errors='coerce')
df_ventas_local['fecha_compra'] = pd.to_datetime(df_ventas_local['fecha_compra'], errors='coerce')

df_clientes_local = df_clientes_local.dropna(subset=['cliente_id', 'provincia'])
df_ventas_local = df_ventas_local.dropna(subset=['transaccion_id', 'cliente_id', 'monto_total_ars'])

print("📊 ¡PROCESAMIENTO PANDAS COMPLETADO EN FILAS CORPORATIVAS!")
print(f"✓ Ingesta exitosa: {len(df_clientes_local)} clientes cargados desde el almacenamiento.")
print(f"✓ Ingesta exitosa: {len(df_ventas_local)} transacciones comerciales procesadas.")


Leyendo archivos CSV desde el disco local...
📊 ¡PROCESAMIENTO PANDAS COMPLETADO EN FILAS CORPORATIVAS!
✓ Ingesta exitosa: 50 clientes cargados desde el almacenamiento.
✓ Ingesta exitosa: 119 transacciones comerciales procesadas.


In [4]:
try:
    conexion = sqlite3.connect(DB_PATH)
    cursor = conexion.cursor()
    
    # Inyección directa de las tablas en el motor relacional
    df_clientes_local.to_sql(name='dim_clientes', con=conexion, if_exists='replace', index=False)
    df_ventas_local.to_sql(name='fact_ventas', con=conexion, if_exists='replace', index=False)
    conexion.commit()
    print("💾 ¡Base de datos relacional creada con tablas 'dim_clientes' y 'fact_ventas'!\n")
    
    # CONSULTA REINA DE AUDITORÍA (Modificado el ORDER BY para priorizar volumen de productos)
    cursor.execute("""
        SELECT c.provincia,
               COUNT(v.transaccion_id) as total_envios,
               SUM(v.cantidad_unidades) as total_productos,
               SUM(v.monto_total_ars) as facturacion,
               AVG(v.monto_total_ars) as ticket
        FROM fact_ventas v
        INNER JOIN dim_clientes c ON v.cliente_id = c.cliente_id
        GROUP BY c.provincia
        ORDER BY total_productos DESC
    """)
    
    # Creamos el DataFrame intermedio con los datos crudos
    columnas_raw = ['Provincia Destino', 'Envíos Despachados', 'Productos Vendidos', 'Facturación Total ($)', 'Ticket Promedio ($)']
    df_auditoria_federal = pd.DataFrame(cursor.fetchall(), columns=columnas_raw)
    
    # TRUCO VISUAL: Forzamos el formato de miles con puntos en las dos columnas de dinero
    for col in ['Facturación Total ($)', 'Ticket Promedio ($)']:
        df_auditoria_federal[col] = df_auditoria_federal[col].apply(lambda x: f"{int(x):,}".replace(",", "."))
    
    print("📊 REPORTE GERENCIAL (SQL NATIVO): Desempeño Logístico y Comercial por Provincia:")
    print("-" * 95)
    display(df_auditoria_federal)
    
except Exception as e:
    print(f"❌ Error crítico en el motor SQL: {e}")
    if 'conexion' in locals(): conexion.rollback()
finally:
    if 'conexion' in locals():
        conexion.close()
        print("\n✓ Conexión a SQLite cerrada de forma segura.")


💾 ¡Base de datos relacional creada con tablas 'dim_clientes' y 'fact_ventas'!

📊 REPORTE GERENCIAL (SQL NATIVO): Desempeño Logístico y Comercial por Provincia:
-----------------------------------------------------------------------------------------------


,Provincia Destino,Envíos Despachados,Productos Vendidos,Facturación Total ($),Ticket Promedio ($)
0,Córdoba,19,44,16.625.000,875.000
1,Entre Ríos,19,42,15.260.000,803.157
2,Río Negro,19,36,9.740.000,512.631
3,Mendoza,17,30,13.015.000,765.588
4,Salta,12,28,5.260.000,438.333
5,Santa Fe,10,21,14.730.000,1.473.000
6,Chaco,11,20,12.795.000,1.163.181
7,Tucumán,8,15,7.145.000,893.125
8,Buenos Aires,4,7,2.675.000,668.750



✓ Conexión a SQLite cerrada de forma segura.
